# SurfBot o'qitish
1. Yuqoridan **Runtime → Change runtime type → T4 GPU** ni tanlang.
2. Har bir katakni tartib bilan ishga tushiring (▶).
3. Oxirida `model.tflite` yuklab olinadi — uni telefondagi SurfBot'ga yuklang.

Har safar **barcha** eski va yangi zip fayllarni birga yuklang: model shunda yaxshilanib boradi.

In [ ]:
from google.colab import files
import zipfile, io, glob, os
os.makedirs('data', exist_ok=True)
up = files.upload()  # surfbot_yozuvlar.zip fayl(lar)ini tanlang
for i, (name, b) in enumerate(up.items()):
    zipfile.ZipFile(io.BytesIO(b)).extractall(f'data/{i}_{name}')
print(len(glob.glob('data/**/*.bin', recursive=True)), 'ta yozuv fayli')

In [ ]:
import numpy as np
W, H = 64, 112
R = 1 + W * H
X, Y = [], []
for p in glob.glob('data/**/*.bin', recursive=True):
    b = np.fromfile(p, dtype=np.uint8)
    n = len(b) // R
    b = b[:n * R].reshape(n, R)
    Y.append(b[:, 0]); X.append(b[:, 1:])
X = np.concatenate(X).reshape(-1, H, W, 1).astype('float32') / 255.0
Y = np.concatenate(Y).astype('int64')
names = ['hech narsa', 'chapga', 'o\'ngga', 'sakrash', 'pastga']
for i, c in enumerate(np.bincount(Y, minlength=5)): print(names[i], c)
print('Jami:', len(Y), '— kamida 3000 ta bo\'lsa yaxshi')

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
rng = np.random.default_rng(0)
idx = rng.permutation(len(Y)); X, Y = X[idx], Y[idx]
counts = np.bincount(Y, minlength=5).astype('float32')
cw = {i: float(len(Y) / (5 * max(c, 1))) for i, c in enumerate(counts)}

m = tf.keras.Sequential([
    tf.keras.Input((H, W, 1)),
    layers.RandomBrightness(0.15, value_range=(0, 1)),
    layers.RandomContrast(0.15),
    layers.Conv2D(24, 5, strides=2, activation='relu', padding='same'),
    layers.Conv2D(48, 3, strides=2, activation='relu', padding='same'),
    layers.Conv2D(64, 3, strides=2, activation='relu', padding='same'),
    layers.Conv2D(96, 3, strides=2, activation='relu', padding='same'),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(5, activation='softmax'),
])
m.compile('adam', 'sparse_categorical_crossentropy', metrics=['accuracy'])
m.fit(X, Y, validation_split=0.15, epochs=40, batch_size=64, class_weight=cw,
      callbacks=[tf.keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True)])

In [ ]:
conv = tf.lite.TFLiteConverter.from_keras_model(m)
open('model.tflite', 'wb').write(conv.convert())
files.download('model.tflite')